# 📝 지식그래프 구축 과제 LV2(응용): JSON 중첩 적재와 관계 속성

> 개념 두 개 이상을 **엮어** 씁니다. `apoc.load.json` 중첩 배열 펼치기, 값을 노드로 빼고 관계 속성에 담기, 관계 속성 집계, `$파라미터` 정렬 조회, **조용한 실패** 재현, 멱등 재적재, 배치 트랜잭션으로 파생 값 만들기, 날짜를 속성과 노드 양쪽에 두고 같은 질문 던지기.

## 풀이 방법
1. 맨 위 **준비 셀들**(연결·초기화·복사·제약)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**에 코드를 채우고 **자가채점 셀**로 확인하세요(✅ 통과!).

- 데이터: `data/seoul_metro_transfer.json`. 지난 과제에서 표로 다뤘던 환승인원을 **중첩 구조 JSON** 으로 받은 것입니다. 요일 세 칸이 `요일별` 배열 안으로 들어가 있습니다.

```json
{"연번": 1, "역명": "신도림",
 "요일별": [{"요일": "평일", "인원": 269275}, {"요일": "토요일", "인원": 229757}, ...]}
```

- **JSON 은 CSV 와 달리 타입이 살아 있습니다.** `인원` 은 이미 숫자라 `toInteger` 가 필요 없습니다.
- 오늘의 모델은 이렇습니다: **`(:Station)-[:TRANSFERS {count}]->(:DayType)`**. 요일을 노드로 뺀 이유는 교안_02 3절의 판단 기준 그대로입니다(같은 값이 여러 행에 반복되고, 그 값으로 묶어 세는 일이 잦다).

화이팅!

> **데이터 출처**: 아래 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다. 교육용으로 지어낸 값이 없습니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | CC0 |
> | 서울교통공사 역간거리·소요시간 (`seoul_metro_stations_cp949.csv`) | 서울 열린데이터광장 OA-12034 | 공공누리 1유형(출처표시) |
> | 서울교통공사 환승역 환승인원 (`seoul_metro_transfer_cp949.csv`) | 서울 열린데이터광장 OA-12033 | 공공누리 1유형(출처표시) |
>
> 의료 그래프는 **2016년에 정리된 자료**입니다. 그래서 "이 약이 이 병에 쓰인다고 **문헌에 정리돼 있다**"까지가 이 데이터가 말하는 것이고, "효능이 입증됐다"는 아닙니다. 지식그래프를 다룰 때 이 구분을 놓치면 안 됩니다.
>
> 지하철 CSV 두 개는 포털에서 받은 **바이트 그대로**라 인코딩이 `CP949` 입니다. UTF-8 로 읽으면 글자가 깨집니다.

아래 준비 셀들을 위에서부터 실행하세요. Neo4j 는 반드시 **실습 전용 DB**에 연결하세요.

이 과제는 **Neo4j Desktop** 에서 풉니다. 아래 제약 셀이 `IS NODE KEY` 를 쓰기 때문에, 그 문법이 막힌 에디션에서는 준비 단계에서 멈춥니다. Desktop 에는 필요한 라이선스가 딸려 옵니다.

파일을 못 읽거나 APOC 를 못 찾는 에러가 나면 `환경_구축_가이드.md` 의 오류 표를 보세요.

In [ ]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 데이터 파일을 Neo4j import 폴더로 복사: 이 셀은 실행만 하세요.
# LOAD CSV·apoc.load.json 은 보안상 서버의 import 폴더 안 파일만 읽습니다.
# - .env 에 NEO4J_IMPORT_DIR 이 있으면 data/ 의 파일을 자동 복사합니다.
# - 없으면(수동 복사한 경우) 그대로 넘어갑니다. Neo4j Desktop 은 인스턴스 메뉴의
#   "Open folder > Import" 로 폴더를 열어 data/ 의 파일을 직접 복사해 두세요.
import shutil
from pathlib import Path

_IMPORT_DIR = os.getenv("NEO4J_IMPORT_DIR", "")
_SRC_DIR = Path("data") if Path("data").exists() else Path("../data")
if _IMPORT_DIR:
    # csv 와 json 만 복사합니다. 다음 준비 셀이 이 파일들을 file:/// 로 읽습니다
    for f in sorted(_SRC_DIR.glob("*.csv")) + sorted(_SRC_DIR.glob("*.json")):
        shutil.copy(f, Path(_IMPORT_DIR) / f.name)
        print("복사:", f.name)
else:
    print("NEO4J_IMPORT_DIR 미설정: data/ 의 파일을 import 폴더에 직접 복사했는지 확인하세요.")

In [ ]:
# [제공 코드] 제약을 먼저 겁니다: 이 셀은 실행만 하세요(적재보다 제약이 먼저입니다).
run_cypher("CREATE CONSTRAINT station_name IF NOT EXISTS "
           "FOR (s:Station) REQUIRE s.name IS NODE KEY")
run_cypher("CREATE CONSTRAINT daytype_name IF NOT EXISTS "
           "FOR (d:DayType) REQUIRE d.name IS NODE KEY")
print("제약 개수:", len(run_cypher("SHOW CONSTRAINTS YIELD name RETURN name")))

## 데이터 살펴보기
아래 셀은 **실행만** 하면 됩니다. 적재할 JSON 의 첫 건을 미리 봅니다(중첩 배열 구조 확인).

In [ ]:
# [제공 코드] 적재 전 JSON 미리보기
preview = run_cypher("CALL apoc.load.json('file:///seoul_metro_transfer.json') YIELD value "
                     "RETURN value LIMIT 1")
print(preview[0]['value'])

## 1. JSON 으로 역 노드 적재하기
**배경**: 공공데이터는 CSV 만큼이나 JSON 으로도 옵니다. `apoc.load.json` 은 배열의 원소를 `value` 로 한 건씩 돌려줍니다. **CSV 와 달리 숫자는 숫자 그대로** 들어옵니다.

**요구사항**:
- `apoc.load.json('file:///seoul_metro_transfer.json')` 로 읽어 `:Station` 노드를 적재하세요.
  - `MERGE` 의 식별 조건은 **`name`**(`역명`)
  - `SET` 으로 **`seq`**(`연번`)를 채웁니다. **변환 함수를 씌우지 마세요**(이미 숫자입니다).
- 적재 후 `Station` 수를 **`n_station`** 에 담으세요.

**예시**: `n_station` 은 **73** 이고, `신도림` 의 `seq` 는 **1** 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- APOC 의 JSON 적재 프로시저를 호출해 한 건씩 받고, 역 이름으로 노드를 MERGE 한 뒤 연번을 SET 한다.

세부구현:
1. CALL 로 JSON 파일을 읽고 YIELD 로 value 를 받는다.
2. value 는 사전이다. 한글 키는 대괄호로 꺼낸다.
3. 역 이름을 식별 조건으로 노드를 MERGE 하고, 연번을 SET 한다.
4. Station 을 세어 n_station 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_station == 73, \
    f'역은 73개여야 합니다(현재 {n_station}). MERGE 의 식별 속성이 name 인지 확인하세요'
seq = run_cypher("MATCH (s:Station {name: '신도림'}) RETURN s.seq AS seq")[0]['seq']
assert seq == 1, f'연번이 정수 1 이어야 합니다(현재 {seq!r}). JSON 값은 이미 숫자입니다'
print('✅ 통과!')

## 2. 중첩 배열을 펼쳐 관계로 만들기
**배경**: `요일별` 은 **배열**입니다. 배열 하나가 관계 세 개가 되어야 합니다. `UNWIND` 로 배열을 행으로 펼치면 원소마다 아래 절이 한 번씩 실행됩니다.

**요구사항**:
- 같은 파일을 다시 읽어 `요일별` 배열을 **`UNWIND`** 로 펼쳐 아래 모델을 만드세요. 이 단원의 도구인 **`apoc.load.json`** 을 권하지만, 파이썬으로 읽어 `$rows` 로 넘겨도 됩니다(채점은 **그래프에 남은 결과**만 봅니다).
  - `:DayType` 노드: 식별 조건은 **`name`**(`요일`)
  - 관계 **`(:Station)-[:TRANSFERS]->(:DayType)`**, 관계 속성 **`count`**(`인원`)
  - 역은 1번에서 이미 만들었으므로 **`MATCH` 로 찾으세요**(`MERGE` 로 쓰면 오타 하나에 새 노드가 생깁니다).
- `DayType` 수를 **`n_daytype`**, `TRANSFERS` 관계 수를 **`n_transfers`** 에 담으세요.

**예시**: `n_daytype` 은 **3**, `n_transfers` 는 **219** 입니다(73개 역 곱하기 요일 3종).

<details><summary>힌트</summary>

```text
접근방법:
- JSON 을 한 건씩 받고, 그 안의 배열을 펼친 뒤, 역은 찾고 요일은 만들고 둘을 관계로 잇는다.

세부구현:
1. CALL 로 JSON 을 읽고 YIELD 로 value 를 받는다.
2. UNWIND 로 value 의 배열 항목을 행으로 펼쳐 이름을 붙인다.
3. 역은 MATCH 로 찾고, 요일 노드는 MERGE 로 만든다.
4. 둘을 MERGE 로 잇고, 관계 변수에 인원 값을 SET 한다.
5. DayType 수와 TRANSFERS 관계 수를 각각 세어 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_daytype == 3, \
    f'요일은 3종이어야 합니다(현재 {n_daytype}). DayType 의 식별 조건이 name 인지 확인하세요'
assert n_transfers == 219, \
    f'관계는 219건이어야 합니다(현재 {n_transfers}). UNWIND 로 배열을 펼쳤는지 확인하세요'
one = run_cypher("MATCH (:Station {name: '신도림'})-[t:TRANSFERS]->(:DayType {name: '평일'}) "
                 "RETURN t.count AS c")[0]['c']
assert one == 269275, \
    f'신도림 의 평일 관계 count 가 269275 여야 합니다(현재 {one})'
print('✅ 통과!')

## 3. 관계 속성으로 집계하기
**배경**: 값이 관계에 있으면 **관계를 잡아 집계**합니다. 노드 속성일 때와 쓰는 자리가 다릅니다.

**요구사항**:
- 요일별 **총 환승인원**을 구해 `{요일: 합계}` 사전 **`by_day`** 에 담으세요.
  - 별칭은 **`요일`**·**`합계`** 로 하고, 합계가 큰 순으로 정렬하세요.
  - 파이썬에서 사전으로 바꾸는 부분까지 직접 쓰세요.
  - 쿼리 문자열은 **`by_day_query`** 에 담고 그것으로 실행하세요. 채점이 같은 쿼리를 한 번 더 돌려 대조합니다.

**예시**: `by_day` 는 `{'평일': 4814846, '토요일': 3861432, '일요일': 2801457}` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 역과 요일을 잇는 관계를 잡아, 요일 이름으로 묶어 관계 속성을 더한다.

세부구현:
1. 관계에 변수를 붙여 MATCH 한다(관계 속성을 써야 하므로).
2. 요일 노드의 이름으로 묶고, 관계의 count 를 합산한다.
3. 합계 내림차순으로 정렬해 반환한다.
4. 결과 행 목록을 순회하며 파이썬 사전으로 만든다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 예시 사전을 그대로 적어도 통과하지 않게, 학생 쿼리를 채점이 다시 돌려 대조한다
_rerun = {r['요일']: r['합계'] for r in run_cypher(by_day_query)}
assert dict(by_day) == _rerun, \
    'by_day 가 by_day_query 를 다시 돌린 결과와 다릅니다. 그 쿼리로 만든 사전인지 확인하세요'
assert 'sum(' in by_day_query.lower(), \
    'by_day_query 에 합계를 내는 집계가 없습니다. 관계의 count 를 sum 으로 더하세요'
_again = {r['d']: r['s'] for r in run_cypher(
    "MATCH (:Station)-[t:TRANSFERS]->(d:DayType) "
    "RETURN d.name AS d, sum(t.count) AS s")}
assert dict(by_day) == _again, \
    f'by_day 가 그래프를 다시 집계한 값과 다릅니다(현재 {dict(by_day)}, 실제 {_again}). '\
    '예시 값을 옮겨 적지 말고 직접 조회하세요'
assert by_day == {'평일': 4814846, '토요일': 3861432, '일요일': 2801457}, \
    f'요일별 합계가 달라졌습니다(현재 {by_day}). 관계 변수에 붙은 count 를 합산했는지 확인하세요'
assert list(by_day) == ['평일', '토요일', '일요일'], \
    f'합계가 큰 순으로 정렬해야 합니다(현재 {list(by_day)})'
print('✅ 통과!')

## 4. 파라미터로 상위 N 뽑기
**배경**: "평일 환승이 가장 많은 다섯 곳" 같은 물음은 **정렬 + 상한**입니다. 상한은 파라미터로 넘겨 같은 쿼리를 재사용합니다.

**요구사항**:
- **평일** 환승인원이 많은 순으로 역 이름 **`$limit`개**를 뽑아 리스트 **`top_names`** 에 담으세요.
- 쿼리 문자열은 **`top_query`** 에 담고 `run_cypher(top_query, limit=...)` 로 실행하세요. 채점이 **같은 쿼리를 다른 상한으로 한 번 더** 돌립니다.
  - `run_cypher` 호출 시 `limit=5` 를 넘깁니다.
  - `LIMIT $limit` 처럼 상한에도 파라미터를 쓸 수 있습니다.
  - 평일만 세야 하므로 `:DayType` 쪽에 `name` 조건을 겁니다.
  - `RETURN` 별칭은 **`역명`** 으로 합니다(채점이 그 이름으로 값을 꺼냅니다).

**예시**: `top_names` 는 `['신도림', '동대문역사문화공원', '고속터미널', '왕십리', '서울역']` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 요일이 평일인 관계만 잡아 역 이름과 인원을 내고, 인원 내림차순으로 정렬해 상한만큼 자른다.

세부구현:
1. 역과 요일을 잇는 관계를 잡되, 요일 노드에 이름 조건을 건다.
2. 역 이름과 관계의 count 를 반환한다.
3. count 내림차순으로 정렬하고 상한을 파라미터로 건다.
4. 결과에서 이름만 뽑아 리스트로 만든다.
5. 쿼리 문자열은 top_query 에 담고 run_cypher(top_query, limit=...) 로 실행한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 예시 리스트를 복사해도 통과하지 않게, 학생 쿼리를 다른 상한으로 채점이 다시 돌린다
_three = [r['역명'] for r in run_cypher(top_query, limit=3)]
assert _three == ['신도림', '동대문역사문화공원', '고속터미널'], \
    f'top_query 를 limit 3 으로 돌리면 상위 세 곳이 나와야 합니다(현재 {_three}). '\
    '상한을 쿼리에 박지 말고 $limit 파라미터로 남겨 두세요'
assert top_names == ['신도림', '동대문역사문화공원', '고속터미널', '왕십리', '서울역'], \
    f'상위 5곳이 순서대로 나와야 합니다(현재 {top_names}). 평일만 골랐는지, 내림차순인지 확인하세요'
print('✅ 통과!')

## 5. 조용한 실패 재현하기
**배경**: 관계 적재에서 **양 끝 노드를 못 찾은 행은 에러 없이 버려집니다.** 레이블을 하나만 잘못 적어도 관계가 0건 만들어지고 쿼리는 "성공"으로 끝납니다. 이걸 직접 겪어 봐야 **왜 건수로 검증하는지** 압니다.

**요구사항**: 같은 적재를 **두 번** 돌립니다. 한 번은 제대로, 한 번은 레이블만 틀리게.
- **(1)** 2번과 같은 적재를 **관계 타입만 `CHECK_TRANSFERS`** 로 바꿔 실행하고, 만들어진 `CHECK_TRANSFERS` 관계 수를 **`n_ok`** 에 담으세요.
- **(2)** 그다음 같은 쿼리에서 역을 찾는 **레이블을 `:Metro`** 로 바꾸고(그런 레이블은 없습니다) 관계 타입도 **`WRONG_TRANSFERS`** 로 바꿔 실행한 뒤, 만들어진 관계 수를 **`n_ghost`** 에 담으세요.
  - 이 **둘째 적재 쿼리**는 **`ghost_q`** 변수에 담아 두세요(자가채점이 같은 쿼리를 한 번 더 돌려 정말 아무것도 안 만드는지 봅니다).
  - 두 **적재 쿼리 모두 에러도 경고도 없이** 끝납니다. `try` 로 감쌀 필요가 없습니다.

**예시**: `n_ok` 는 **219**, `n_ghost` 는 **0** 입니다. 두 쿼리는 **레이블 한 단어**만 다릅니다.

> (2)를 세는 쿼리를 돌리면 그제야 `The relationship type WRONG_TRANSFERS does not exist` 라는 **경고**가 뜹니다. 만들어진 것이 하나도 없어서 그 타입이 아예 없기 때문입니다. 이건 에러가 아니라 참고 문구라 **실행을 멈추지 않습니다.** 셀 출력에 섞여 나와도 놀라지 마세요(자가채점 셀에서도 같은 이유로 한 번 더 뜹니다).

<details><summary>힌트</summary>

```text
접근방법:
- 2번 쿼리를 두 벌 복사해, 한 벌은 관계 타입만 바꾸고 다른 한 벌은 레이블까지 바꿔 실행한 뒤 각각 만들어진 관계 수를 센다.

세부구현:
1. 2번의 적재 쿼리를 그대로 쓰되 관계 타입만 지문의 첫 이름으로 바꿔 실행한다.
2. 그 타입의 관계 수를 세어 n_ok 에 담는다.
3. 같은 쿼리에서 MATCH 의 레이블을 존재하지 않는 것으로, 관계 타입을 지문의 둘째 이름으로 바꿔 ghost_q 에 담고 실행한다.
4. 그 타입의 관계 수를 세어 n_ghost 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_ok == 219, \
    f'레이블이 맞으면 219건이 만들어집니다(현재 {n_ok}). 역을 :Station 으로 찾았는지 확인하세요'
assert n_ghost == 0, \
    f'0 이어야 합니다(현재 {n_ghost}). 레이블을 :Metro 로 바꿨는지 확인하세요'
# 두 수를 그래프에서 다시 센다. 적재를 돌리지 않고 값만 적어 넣었다면 여기서 걸린다
made = run_cypher("MATCH (:Station)-[t:CHECK_TRANSFERS]->(:DayType) RETURN count(t) AS n")[0]['n']
assert made == 219, \
    f'그래프에 CHECK_TRANSFERS 관계가 219건 있어야 합니다(현재 {made}). (1) 적재를 실제로 실행했는지 확인하세요'
ghost = run_cypher("MATCH ()-[t:WRONG_TRANSFERS]->() RETURN count(t) AS n")[0]['n']
assert ghost == 0, \
    f'WRONG_TRANSFERS 관계는 하나도 없어야 합니다(현재 {ghost}). 레이블을 :Metro 로 바꿨는지 확인하세요'
still = run_cypher("MATCH (:Station)-[t:TRANSFERS]->(:DayType) RETURN count(t) AS n")[0]['n']
assert still == 219, \
    f'원래 관계 219건은 그대로여야 합니다(현재 {still})'
# 여기까지는 (2)를 아예 안 돌리고 n_ghost = 0 이라고 적어도 다 통과한다.
# 그래서 담아 둔 둘째 쿼리를 채점이 직접 한 번 돌려 본다
assert ':Metro' in ghost_q, \
    'ghost_q 에서 역을 찾는 레이블을 :Metro 로 바꿔야 합니다'
assert 'WRONG_TRANSFERS' in ghost_q, \
    'ghost_q 의 관계 타입을 WRONG_TRANSFERS 로 바꿔야 합니다'
run_cypher(ghost_q)
ghost2 = run_cypher("MATCH ()-[t:WRONG_TRANSFERS]->() RETURN count(t) AS n")[0]['n']
assert ghost2 == 0, \
    f'ghost_q 를 다시 돌려도 0 이어야 합니다(현재 {ghost2}). 레이블을 :Metro 로 바꿨는지 확인하세요'
print('✅ 통과!')

## 6. 멱등성 확인: 관계도 늘지 않는다
**배경**: 노드뿐 아니라 **관계도** `MERGE` 로 만들면 재적재해도 늘지 않습니다. 그리고 `SET` 이 값을 원본으로 다시 덮어씁니다.

**아래 제공 셀이 모든 역의 평일 관계 `count` 를 0 으로 망가뜨립니다.** 그다음 문제를 푸세요.

**요구사항**:
- **2번과 똑같은 적재**를 한 번 더 실행한 뒤, `TRANSFERS` 관계 수를 **`n_again`** 에 담으세요.

**예시**: `n_again` 은 여전히 **219** 이고, 평일 인원 합계가 **4,814,846** 로, `신도림` 의 값이 **269,275** 로 되돌아옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 2번의 적재를 그대로 한 번 더 실행하고, 관계 수를 다시 센다.

세부구현:
1. 2번과 똑같은 적재 쿼리를 다시 실행한다.
2. TRANSFERS 관계 수를 세어 n_again 에 담는다.
```

</details>

In [ ]:
# [제공 코드] 모든 역의 평일 환승인원을 0 으로 망가뜨립니다. 이 셀은 실행만 하세요.
# (한 곳만 망가뜨리면 그 한 값만 손으로 되돌려도 다음 채점을 통과해 버립니다)
# 이 셀을 다시 실행하면 값이 또 0 이 됩니다. 그때는 6번 답안 셀을 한 번 더 실행하세요
run_cypher("MATCH (:Station)-[t:TRANSFERS]->(:DayType {name: '평일'}) "
           "SET t.count = 0")
# 집계(count·sum)로 받으면 대상이 없어도 한 행이 돌아온다(빈 결과에서 터지지 않게)
broken = run_cypher("MATCH (:Station)-[t:TRANSFERS]->(:DayType {name: '평일'}) "
                    "RETURN count(t) AS n, sum(t.count) AS c")[0]
print(f"평일 환승인원을 {broken['c']} 으로 바꿨습니다(대상 {broken['n']}건).")

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert n_again == 219, \
    f'재적재해도 219건이어야 합니다(현재 {n_again}). 관계도 MERGE 로 만들었는지 확인하세요'
# 한 값만 되돌려도 통과하지 않게 평일 합계를 함께 본다
weekday_sum = run_cypher("MATCH (:Station)-[t:TRANSFERS]->(:DayType {name: '평일'}) "
                        "RETURN sum(t.count) AS c")[0]['c']
assert weekday_sum == 4814846, \
    f'평일 합계가 {weekday_sum} 입니다. 2번과 똑같은 적재를 다시 실행해 값을 되돌리세요'
fixed = run_cypher("MATCH (:Station {name: '신도림'})-[t:TRANSFERS]->(:DayType {name: '평일'}) "
                   "RETURN t.count AS c")[0]['c']
assert fixed == 269275, \
    f'269275 로 되돌아와야 합니다(현재 {fixed}). 2번과 똑같은 적재를 다시 실행했는지 확인하세요'
print('✅ 통과!')

## 7. 배치 트랜잭션으로 파생 값 만들기
**배경**: 이미 적재한 그래프에서 **계산해 만든 값**을 노드에 저장해 두면 다음 조회가 간단해집니다. 대상이 많을 때는 `IN TRANSACTIONS` 로 끊어 커밋합니다.

**요구사항**:
- 각 역의 **주말 비율**(`토요일 인원 / 평일 인원`)을 소수 셋째 자리까지 반올림해 `:Station` 의 **`sat_ratio`** 속성에 저장하세요.
  - 역을 잡아 **배치 블록으로 들여보내** 갱신하세요(교안_03 3절 마지막 시연과 같은 형태). 배치 크기는 **20행**입니다.
  - 중괄호 안에서 그 역의 평일 관계와 토요일 관계를 각각 잡아 나눕니다.
  - 정수끼리 나누면 정수가 되니 **`toFloat`** 를 한쪽에 씌우고, `round(x, 3)` 으로 반올림합니다.
- 갱신 쿼리는 **`ratio_q`** 변수에 담고 `run_cypher(ratio_q)` 로 실행하세요(자가채점이 그 쿼리가 정말 배치 블록을 쓰는지 봅니다).
- 채운 뒤 `sat_ratio` 가 있는 역 수를 **`n_ratio`**, 비율이 가장 높은 역 이름을 **`top_ratio`** 에 담으세요.

**예시**: `n_ratio` 는 **73**, `top_ratio` 는 **`청량리`**(비율 1.125) 입니다. 평일보다 토요일에 더 붐비는 유일한 역입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 역을 하나씩 잡아 배치 블록 안으로 들여보내고, 그 안에서 두 요일의 관계를 잡아 나눈 값을 저장한다.

세부구현:
1. 역을 MATCH 한다.
2. CALL 뒤 괄호에 그 역 변수를 적어 배치 블록으로 들여보낸다.
3. 블록 안에서 그 역의 평일 관계와 토요일 관계를 각각 MATCH 한다.
   3-1. 한쪽에 실수 변환을 씌워 나눈 뒤 반올림해 SET 한다.
4. 블록 뒤에 배치 크기 절을 붙인다.
5. 그 쿼리 문자열을 ratio_q 에 담고 run_cypher(ratio_q) 로 실행한다.
6. sat_ratio 가 비어 있지 않은 역을 세고, 비율 내림차순 첫 역의 이름을 꺼낸다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 배치 블록 없이 한 번에 SET 해도 값은 같다. 그래서 쿼리 자체가 그 문법을 쓰는지 본다
assert 'IN TRANSACTIONS' in ratio_q.upper(), \
    'ratio_q 에 CALL (s) { ... } IN TRANSACTIONS 블록을 쓴 쿼리를 담으세요'
import re as _re
assert _re.search(r'IN\s+TRANSACTIONS\s+OF\s+20\s+ROWS', ratio_q, _re.I), \
    '배치 크기는 20행입니다. IN TRANSACTIONS OF 20 ROWS 로 적었는지 확인하세요'
# 학생 변수만 보면 한 곳만 채워도 통과한다. 그래프에서 다시 센다
_filled = run_cypher("MATCH (s:Station) WHERE s.sat_ratio IS NOT NULL "
                     "RETURN count(s) AS n")[0]['n']
assert _filled == 73, \
    f'sat_ratio 가 채워진 역이 {_filled}곳입니다. 73곳 모두 채워야 합니다'
assert n_ratio == 73, \
    f'73곳이어야 합니다(현재 {n_ratio}). 배치 블록 안에서 두 요일 관계를 모두 잡았는지 확인하세요'
assert top_ratio == '청량리', \
    f'주말 비율 1위는 청량리 입니다(현재 {top_ratio}). 토요일을 평일로 나눴는지 확인하세요'
value = run_cypher("MATCH (s:Station {name: '청량리'}) RETURN s.sat_ratio AS r")[0]['r']
assert abs(value - 1.125) < 0.0005, \
    f'비율이 1.125 여야 합니다(현재 {value}). round(..., 3) 을 썼는지 확인하세요'
print('✅ 통과!')

## 8. 기준일을 날짜 노드로 올리고 해를 얹기
**배경**: 이 JSON 도 **2025-11-30 기준** 자료입니다(포털 원본 이름이 `서울교통공사_환승역환승인원정보_20251130.csv`). 2번에서 요일을 노드로 뺐던 것과 **같은 판단**을 날짜에도 대 봅니다. 다만 날짜는 속성으로 두었을 때 잘되는 일도 있어서, 여기서는 **두 가지를 다** 해 두고 같은 질문을 두 방식으로 던져 봅니다.

**요구사항**:
1. 모든 `:Station` 에 기준일을 **`surveyed`** 속성으로 넣으세요(2025-11-30, **날짜 자료형**).
2. 같은 기준일을 **`:SurveyDay`** 노드(속성 이름 **`date`**, 항목 1과 같은 **날짜 자료형**)로도 만들고, 모든 역을 **`SURVEYED_ON`** 관계로 그 노드에 이으세요(**역에서 날짜 쪽** 방향, `MERGE` 로).
3. **같은 날 조사된 역 쌍**의 수를 두 방식으로 세어 각각 **`n_pairs_prop`**(속성 값을 견주는 방식)과 **`n_pairs_node`**(가운데 날짜 노드를 함께 가리키는 방식)에 담으세요. 같은 쌍이 앞뒤 바뀌어 두 번 세어지지 않도록 **역 이름에 순서를 주는 조건**을 넣습니다.
   - 두 쿼리 문자열은 **`pairs_prop_q`**·**`pairs_node_q`** 에 담고 그것으로 실행하세요. 채점이 두 쿼리를 다시 돌려 대조합니다.
   - 두 쿼리의 `RETURN` 별칭은 **`n`** 으로 합니다(채점이 그 이름으로 값을 꺼냅니다).
4. 날짜 노드 위에 **`:SurveyYear`** 노드(속성 이름 **`year`**)를 얹어 **`HAS_DAY`** 관계로 이으세요(**해에서 날 쪽** 방향). 연도 값은 손으로 적지 말고 **날짜 노드에서 꺼내** 씁니다.
   - 자료 시점이 하나뿐이라 **달 층은 생략하고 해에서 날로 바로 잇습니다**. 달 노드를 만들면 이 그래프에 쓰이지 않는 노드가 남습니다.
5. 해에서 날로 내려가는 경로를 따라 **해마다 조사된 역이 몇 곳인지** 구해 **`by_year`** 에 담으세요. RETURN 별칭은 **`year`** 와 **`n`**, 정렬은 연도 오름차순입니다.

**예시**: `n_pairs_prop` 과 `n_pairs_node` 는 둘 다 **2628** 입니다(역 73곳에서 두 곳을 고르는 짝의 수). `by_year` 는 **`[{'year': 2025, 'n': 73}]`** 입니다.

> 지금 자료는 기준일이 하나뿐이라 "같은 날"과 "같은 해"의 답이 같습니다. 다른 시점 자료가 쌓이면 그때부터 갈라집니다. 여기서는 **모양을 손에 익히는 것**이 목적입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 기준일을 두 자리에 둔다. 하나는 역의 속성으로, 하나는 노드로.
- 같은 날 짝짓기는 속성이면 값이 같은지 견주고, 노드면 가운데 노드를 함께 가리키도록 그린다.

세부구현:
1. 모든 역을 MATCH 해 기준일 속성을 SET 한다. 날짜는 날짜를 만드는 함수로 감싼다.
2. 날짜 노드를 MERGE 로 하나 만들고 WITH 로 넘긴 뒤, 모든 역을 그 노드로 MERGE 해 잇는다.
3. 짝 세기
   3-1. 속성 방식: 역 둘을 각각 잡아 기준일이 같고 이름 순서가 앞뒤인 것만 센다.
   3-2. 노드 방식: 두 역이 가운데 날짜 노드 하나를 함께 가리키는 패턴을 그리고, 이름 순서 조건만 남긴다.
4. 날짜 노드를 MATCH 해 그 날짜에서 연도를 꺼내 연도 노드를 MERGE 하고 날짜 쪽으로 잇는다.
5. 연도에서 날로 내려와 그 날에 이어진 역을 세고, 연도 오름차순으로 정렬해 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 값을 손으로 적어도 통과하지 않게, 두 쿼리를 채점이 다시 돌린다
assert run_cypher(pairs_prop_q)[0]['n'] == n_pairs_prop, \
    'n_pairs_prop 이 pairs_prop_q 를 다시 돌린 값과 다릅니다'
assert run_cypher(pairs_node_q)[0]['n'] == n_pairs_node, \
    'n_pairs_node 가 pairs_node_q 를 다시 돌린 값과 다릅니다'
assert 'surveyed' in pairs_prop_q, \
    '속성 방식은 두 역의 surveyed 값을 견주어야 합니다'
assert 'SURVEYED_ON' in pairs_node_q, \
    '노드 방식은 가운데 날짜 노드를 함께 가리키는 패턴(SURVEYED_ON)이어야 합니다'
assert n_pairs_prop == 2628, \
    f'속성 방식은 2628 쌍이어야 합니다(현재 {n_pairs_prop}). 이름 순서 조건을 넣었는지 확인하세요'
assert n_pairs_node == 2628, \
    f'노드 방식도 2628 쌍이어야 합니다(현재 {n_pairs_node}). 두 역이 같은 날짜 노드를 함께 가리키는지 확인하세요'
n_rel = run_cypher("MATCH ()-[r:SURVEYED_ON]->() RETURN count(r) AS n")[0]['n']
assert n_rel == 73, \
    f'SURVEYED_ON 관계는 역마다 하나씩 73개여야 합니다(현재 {n_rel}). 모든 역을 이었는지 확인하세요'
n_day = run_cypher("MATCH (d:SurveyDay) RETURN count(d) AS n")[0]['n']
assert n_day == 1, \
    f'기준일 노드는 1개여야 합니다(현재 {n_day}). 역마다 따로 만들지 말고 하나를 함께 가리키게 하세요'
n_year = run_cypher("MATCH (y:SurveyYear) RETURN count(y) AS n")[0]['n']
assert n_year == 1, \
    f'연도 노드는 1개여야 합니다(현재 {n_year}). 날짜 노드 위에 연도 노드를 얹어 이었는지 확인하세요'
# by_year 만 보면 트리를 안 만들고 아무 쿼리로나 같은 값을 낼 수 있다.
# 그래서 해 -> 날 -> 역 경로를 채점이 직접 한 번 타 본다(방향까지 함께 본다)
tree = run_cypher("MATCH (y:SurveyYear)-[:HAS_DAY]->(:SurveyDay)"
                  "<-[:SURVEYED_ON]-(s:Station) "
                  "RETURN y.year AS year, count(s) AS n ORDER BY year")
assert tree == [{'year': 2025, 'n': 73}], \
    f'해에서 날로 내려가는 경로가 이어져 있어야 합니다(현재 {tree}). '  \
    f'달 층을 끼우지 말고 해에서 날로 바로 이었는지, 그리고 '  \
    f'SurveyYear-[:HAS_DAY]->SurveyDay 와 Station-[:SURVEYED_ON]->SurveyDay 의 방향을 확인하세요'
assert by_year == [{'year': 2025, 'n': 73}], \
    f'by_year 는 [{{\'year\': 2025, \'n\': 73}}] 이어야 합니다(현재 {by_year}). 별칭을 year·n 으로 맞췄는지 확인하세요'
# 날짜를 글자로 넣으면 크기 비교가 에러 없이 빈 결과가 된다. 그래서 여기서 걸린다
typed = run_cypher("MATCH (s:Station) WHERE s.surveyed >= date('2025-01-01') "
                   "RETURN count(s) AS n")[0]['n']
assert typed == 73, \
    f'기준일을 날짜 자료형으로 넣어야 합니다(크기 비교에 걸린 역 {typed}곳). 따옴표만 씌운 글자가 아닌지 확인하세요'
print('✅ 통과!')